# 02 — Modeling
Semua langkah (feature engineering → preprocessing → model) berada dalam **satu `Pipeline`** (`scripts/train.py::build_pipeline`). Model: `HistGradientBoostingClassifier` (bukan Neural Network).

In [ ]:
import sys
sys.path.append("..")  # agar package `scripts` bisa di-import dari folder notebooks/

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scripts import config
from scripts.preprocess import load_train, load_test, load_metadata, split_xy

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from scripts.train import build_pipeline
from scripts.evaluate import micro_f1, report, plot_confusion

## Split & baseline
Memakai sampel 100k baris agar notebook cepat. Training penuh dilakukan via `python -m scripts.train`.

In [ ]:
df = load_train().sample(100_000, random_state=config.RANDOM_STATE)
X, y = split_xy(df)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=config.RANDOM_STATE)

baseline = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print("Baseline (kelas mayoritas) micro-F1:", round(micro_f1(y_va, baseline.predict(X_va)), 4))

## Pipeline

In [ ]:
pipe = build_pipeline()
pipe

In [ ]:
pipe.fit(X_tr, y_tr)
pred = pipe.predict(X_va)
print("Pipeline micro-F1:", round(micro_f1(y_va, pred), 4))
report(y_va, pred)

In [ ]:
plot_confusion(y_va, pred, save_path=config.FIGURES_DIR / "confusion_matrix_valid.png")
from IPython.display import Image
Image(config.FIGURES_DIR / "confusion_matrix_valid.png")

## Tuning (opsional, lambat)
Tuning memakai `RandomizedSearchCV` di atas pipeline yang sama:
```bash
python -m scripts.tune --n-iter 10 --sample 100000
python -m scripts.train --params models/best_params.json
```

## Training final
```bash
python -m scripts.train        # CV + fit penuh + simpan models/pipeline_model_hgb.pkl
```